## Visualize the data created by the cpp examples

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import Normalize, ListedColormap
from matplotlib.colorbar import ColorbarBase
import matplotlib.image as mpimg
from pylupnt import plasma as pecsimpy

In [ ]:
BASEPATH = pecsimpy.get_plasma_base_path()
TESTPATH = BASEPATH + "/output/test"
DATA_DIR_CPP = TESTPATH + "/cpp/csv"
OUTPUT_DIR_CPP = TESTPATH + "/cpp/images"
DATA_DIR_FORTRAN = TESTPATH + "/fortran/csv"
OUTPUT_DIR_FORTRAN = TESTPATH + "/fortran/images"
OUTPUT_DIR_GCPM = TESTPATH + "/orig/images/"

if not os.path.exists(OUTPUT_DIR_CPP):
    os.makedirs(OUTPUT_DIR_CPP)

if not os.path.exists(OUTPUT_DIR_FORTRAN):
    os.makedirs(OUTPUT_DIR_FORTRAN)

print("DATA_DIR_CPP: ", DATA_DIR_CPP)

In [ ]:
## Load the example data from the cloud

load_data = False  # <---- Change this to True if you want to load the data from the cloud and run the simulation without having to run the C++ code examples first.

if load_data:
    import gdown
    import zipfile

    DL_DIR = BASEPATH + "/output/dl"
    if not os.path.exists(DL_DIR):
        os.makedirs(DL_DIR)

    DATA_DIR_FORTRAN = BASEPATH + "/output/dl/output/test/fortran/csv"
    DATA_DIR_CPP = BASEPATH + "/output/dl/output/test/cpp/csv"
    OUTPUT_DIR_GCPM = BASEPATH + "/output/dl/output/test/orig/images/"

    gdfile_id = "1OzKIEitpD4i0Ri_kY2Zm7P29g6Is32uQ"
    gdfile_link = "https://drive.google.com/uc?id=" + gdfile_id

    # load output zip file and place it to '../output_dl'
    gdown.download(gdfile_link, "output_dl.zip", quiet=False)

    # unzip the file
    with zipfile.ZipFile("output_dl.zip", "r") as zip_ref:
        zip_ref.extractall(DL_DIR)

    # remove the zip file
    os.remove("output_dl.zip")

## Example1: 1D Along a Field
- 1D Along a field line 
- Run examples/cpp/field_aligned.cpp before running this cell
- Should match with fortran/gcpm_v24/gcpm_v24_fieldaligned.jpg

In [ ]:
# Load the CSV file
def show_gcpm_image(filename, size=(10, 10)):
    fig, ax = plt.subplots(figsize=size)
    img = mpimg.imread(OUTPUT_DIR_GCPM + filename)
    imgplot = plt.imshow(img)
    plt.axis("off")
    plt.show()
    plt.close()


def plot_field_aligned(DATA_DIR, OUTPUT_DIR, version):
    input_csv = os.path.join(DATA_DIR, "test_fieldaligned.csv")
    data = pd.read_csv(
        input_csv, header=None
    ).values  # Assuming the CSV file has no header

    # Extract columns
    lats = data[:, 0]
    rs = data[:, 1]
    ns = data[:, 2]

    # Convert radius to altitude in km
    altitudes = (rs - 1.0) * 6371.0

    # Visualization
    notation = "GCPM Version 2.4 (IRI 2007 + GCPM v2.4 {})".format(version)
    plt.figure(figsize=(6, 4))
    plt.plot(altitudes, ns, color="black", linewidth=2)

    # Configure plot settings
    plt.xscale("linear")
    plt.yscale("log")
    plt.xlim(0.0, 700.0)
    plt.ylim(1.0e2, 1.0e6)
    plt.xlabel("Altitude (km)", fontsize=12, weight="bold")
    plt.ylabel("Density (cm$^{-3}$)", fontsize=12, weight="bold")
    plt.title(notation, fontsize=12, weight="bold")
    plt.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.7)

    # Save the plot as a JPEG image
    output_image = os.path.join(OUTPUT_DIR, "gcpm_v24_fieldaligned.jpg")
    plt.savefig(output_image, dpi=300, bbox_inches="tight")
    plt.show()


plot_field_aligned(DATA_DIR_CPP, OUTPUT_DIR_CPP, "C++")
plot_field_aligned(DATA_DIR_FORTRAN, OUTPUT_DIR_FORTRAN, "Fortran")

# show the image orig/images/gcpm_v24_fieldaligned.jpg
show_gcpm_image("gcpm_v24_fieldaligned.jpg", size=(7, 7))

## Example3: 2D Equatorial Slice
- Run examples/cpp/equatorial_slice.cpp before running this cell
- This should generate a similar image as gcpm_v24_equaotrial.jpg in fortran/gcpm_v24 

In [ ]:
def plot_equatorial(DATA_DIR, OUTPUT_DIR, version):
    input_csv = os.path.join(DATA_DIR, "test_equatorial.csv")
    data = pd.read_csv(
        input_csv, header=None
    ).values  # Assuming the CSV file has no header

    # Process the data to apply log10 transformation
    lden = np.full_like(
        data, -2.0, dtype=float
    )  # Initialize with -2.0 for invalid data
    positive_indices = np.where(data > 0.0)
    lden[positive_indices] = np.log10(data[positive_indices])

    # Set visualization parameters
    minscale = -2.0
    maxscale = 6.0
    notation = "GCPM Version 2.4 (IRI 2007 + GCPM v2.4 {})".format(version)

    # define a colar map where default is rainbow, but the lower end is black
    # Create a custom colormap
    original_cmap = plt.cm.jet
    colors = original_cmap(np.linspace(0, 1, 256))
    colors[0] = [0, 0, 0, 1]  # Set the lowest value to black (RGBA)
    custom_cmap = ListedColormap(colors)

    # Plot the density data
    fig, ax = plt.subplots(figsize=(6, 4))
    cax = ax.imshow(
        lden.T,
        cmap=custom_cmap,
        origin="lower",
        extent=[-10.0, 10.0, -10.0, 10.0],
        norm=Normalize(vmin=minscale, vmax=maxscale),
    )

    # Configure plot appearance
    ax.set_title(notation, fontsize=12, weight="bold")
    ax.set_xlabel("SM X-Axis (R$_E$)", fontsize=12, weight="bold")
    ax.set_ylabel("SM Y-Axis (R$_E$)", fontsize=12, weight="bold")
    ax.tick_params(axis="both", which="major", labelsize=10, width=1.5)
    # ax.scatter(2.0, 8.0, color="red", marker="*", s=10, label="Equatorial Point")

    # Add the colorbar
    cbar = fig.colorbar(cax, ax=ax, orientation="vertical", fraction=0.046, pad=0.04)
    cbar.set_label(r"log$_{10}$(density [cm$^{-3}$])", fontsize=12, weight="bold")
    cbar.ax.tick_params(labelsize=10)

    # Save the plot as a JPEG image
    output_image = os.path.join(OUTPUT_DIR, "gcpm_v24_equatorial.jpg")
    plt.savefig(output_image, dpi=300, bbox_inches="tight")
    plt.show()


plot_equatorial(DATA_DIR_CPP, OUTPUT_DIR_CPP, "C++")
plot_equatorial(DATA_DIR_FORTRAN, OUTPUT_DIR_FORTRAN, "Fortran")
show_gcpm_image("gcpm_v24_equatorial.jpg", size=(6, 6))

## Example 4: 2D Meridianal Slice
- Run examples/cpp/meridianal_slice.cpp before running this cell (It could take pretty long)
- Should generate a similiar figures as gcpm_v24_meridian_{}h_{}h_kp1p0.jpg in fortran/gcpm_v24

In [ ]:
def plot_meridian(DATA_DIR, OUTPUT_DIR, version):
    nfiles = 12

    fig, axs = plt.subplots(nfiles // 3, 3, figsize=(12, 15))
    fig.suptitle(
        "GCPM Version 2.4 (IRI 2007 + GCPM v2.4 {})".format(version),
        fontsize=14,
        weight="bold",
    )

    for i in range(nfiles):
        mlt_n = i
        mlt_p = i + 12
        input_csv = os.path.join(
            DATA_DIR, "gcpm_v24_meridian_{}h_{}h_kp1p0.csv".format(mlt_n, mlt_p)
        )
        data = pd.read_csv(
            input_csv, header=None
        ).values  # Assuming the CSV file has no header

        # Process the data to apply log10 transformation
        lden = np.full_like(
            data, -2.0, dtype=float
        )  # Initialize with -2.0 for invalid data
        positive_indices = np.where(data > 0.0)
        lden[positive_indices] = np.log10(data[positive_indices])

        # Set visualization parameters
        minscale = -2.0
        maxscale = 6.0
        notation = "Meridian Plane at {}-{} MLT".format(mlt_n, mlt_p)

        # define a colar map where default is rainbow, but the lower end is black
        # Create a custom colormap
        original_cmap = plt.cm.jet
        colors = original_cmap(np.linspace(0, 1, 256))
        colors[0] = [0, 0, 0, 1]  # Set the lowest value to black (RGBA)
        custom_cmap = ListedColormap(colors)

        # Plot the density data
        ax = axs[i // 3, i % 3]
        cax = ax.imshow(
            lden.T,
            cmap=custom_cmap,
            origin="lower",
            extent=[-10.0, 10.0, -10.0, 10.0],
            norm=Normalize(vmin=minscale, vmax=maxscale),
        )

        # Configure plot appearance
        ax.set_title(notation, fontsize=12, weight="bold")
        ax.set_xlabel("SM Equatorial (R$_E$)", fontsize=12)
        ax.set_ylabel("SM Z-Axis (R$_E$)", fontsize=12)
        ax.tick_params(axis="both", which="major", labelsize=10, width=1.5)

        # Add the colorbar
        cbar = fig.colorbar(
            cax, ax=ax, orientation="vertical", fraction=0.046, pad=0.04
        )
        cbar.set_label(r"log$_{10}$(density [cm$^{-3}$])", fontsize=12, weight="bold")
        cbar.ax.tick_params(labelsize=10)

    # format figure
    plt.tight_layout()
    output_image = os.path.join(OUTPUT_DIR, "gcpm_v24_meridian.jpg")
    plt.savefig(output_image, dpi=300, bbox_inches="tight")
    plt.show()


def show_gcpm_image_meridian():
    nfiles = 12
    fig, axs = plt.subplots(nfiles // 3, 3, figsize=(12, 15))

    for i in range(nfiles):
        mlt_n = i
        mlt_p = i + 12

        if mlt_n < 10:
            mlt_n_str = "0" + str(mlt_n)
        else:
            mlt_n_str = str(mlt_n)
        mlt_p_str = str(mlt_p)

        img = mpimg.imread(
            OUTPUT_DIR_GCPM
            + "gcpm_v24_meridian_{}h_{}h_kp1p0.jpg".format(mlt_n_str, mlt_p_str)
        )
        ax = axs[i // 3, i % 3]
        ax.imshow(img)
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR_GCPM + "gcpm_v24_meridian.jpg", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


plot_meridian(DATA_DIR_CPP, OUTPUT_DIR_CPP, "C++")
plot_meridian(DATA_DIR_FORTRAN, OUTPUT_DIR_FORTRAN, "Fortran")
show_gcpm_image_meridian()

## Creating 3D Animation

In [ ]:
import matplotlib.animation as animation


def plot_meridian_each(DATA_DIR, OUTPUT_DIR, version):
    nfiles = 12

    for i in range(nfiles):
        mlt_n = i
        mlt_p = i + 12
        input_csv = os.path.join(
            DATA_DIR, "gcpm_v24_meridian_{}h_{}h_kp1p0.csv".format(mlt_n, mlt_p)
        )
        data = pd.read_csv(
            input_csv, header=None
        ).values  # Assuming the CSV file has no header

        # Process the data to apply log10 transformation
        lden = np.full_like(
            data, -2.0, dtype=float
        )  # Initialize with -2.0 for invalid data
        positive_indices = np.where(data > 0.0)
        lden[positive_indices] = np.log10(data[positive_indices])

        # Set visualization parameters
        minscale = -2.0
        maxscale = 6.0
        notation = "Meridian Plane at {}-{} MLT".format(mlt_n, mlt_p)

        # define a colar map where default is rainbow, but the lower end is black
        # Create a custom colormap
        original_cmap = plt.cm.jet
        colors = original_cmap(np.linspace(0, 1, 256))
        colors[0] = [0, 0, 0, 1]  # Set the lowest value to black (RGBA)
        custom_cmap = ListedColormap(colors)

        # Plot the density data
        fig, ax = plt.subplots(figsize=(4, 4))
        cax = ax.imshow(
            lden.T,
            cmap=custom_cmap,
            origin="lower",
            extent=[-10.0, 10.0, -10.0, 10.0],
            norm=Normalize(vmin=minscale, vmax=maxscale),
        )

        # Configure plot appearance
        ax.set_title(notation, fontsize=12, weight="bold")
        ax.set_xlabel("SM Equatorial (R$_E$)", fontsize=12)
        ax.set_ylabel("SM Z-Axis (R$_E$)", fontsize=12)
        ax.tick_params(axis="both", which="major", labelsize=10, width=1.5)

        # Add the colorbar
        cbar = fig.colorbar(
            cax, ax=ax, orientation="vertical", fraction=0.046, pad=0.04
        )
        cbar.set_label(r"log$_{10}$(density [cm$^{-3}$])", fontsize=12)
        cbar.ax.tick_params(labelsize=10)

        output_image = os.path.join(
            OUTPUT_DIR, "gcpm_v24_meridian_{}h_{}h_kp1p0.jpg".format(mlt_n, mlt_p)
        )
        plt.savefig(output_image, dpi=300, bbox_inches="tight")


plot_meridian_each(DATA_DIR_CPP, OUTPUT_DIR_CPP, "C++")

In [ ]:
import glob
from PIL import Image

image_array = []
for i in range(12):
    mlt_n = i
    mlt_p = i + 12
    my_file = OUTPUT_DIR_CPP + "/gcpm_v24_meridian_{}h_{}h_kp1p0.jpg".format(
        mlt_n, mlt_p
    )
    image = Image.open(my_file)
    image_array.append(image)

print("image_arrays shape:", np.array(image_array).shape)

# Create the figure and axes objects
fig, ax = plt.subplots()
plt.axis("off")

# Set the initial image
im = ax.imshow(image_array[0], animated=True)


def update(i):
    im.set_array(image_array[i])
    # clear obj
    return (im,)


# Create the animation object
animation_fig = animation.FuncAnimation(
    fig,
    update,
    frames=len(image_array),
    interval=200,
    blit=True,
    repeat_delay=10,
)

# Show the animation
plt.show()

animation_fig.save(OUTPUT_DIR_CPP + "/gcpm_v24_meridian.gif", writer="pillow", fps=5)

### 3D Plotin mayavi

In [ ]:
from scipy.interpolate import griddata

# (X, Y, Z) of the data points
[y_plane, x_plane] = np.meshgrid(np.linspace(-10, 10, 201), np.linspace(-10, 10, 201))

# print(x_plane.flatten())
# print(y_plane.flatten())

ref_pos = np.array([])
den_vec = np.array([])

ldens = []
for i in range(12):
    # rotation around z axis
    theta = np.pi / 12 * (2 * i)
    x_vec = x_plane.flatten() * np.cos(theta)
    y_vec = x_plane.flatten() * np.sin(theta)
    z_vec = y_plane.flatten()
    data = pd.read_csv(
        os.path.join(
            DATA_DIR_CPP, "gcpm_v24_meridian_{}h_{}h_kp1p0.csv".format(i, i + 12)
        ),
        header=None,
    ).values
    lden = np.full_like(data, -2.0, dtype=float)[:, :-1]  # (201, 201)
    positive_indices = np.where(data > 0.0)
    lden[positive_indices] = np.log10(data[positive_indices])
    lden = lden.flatten()
    ldens.append(lden)

    posvec = np.vstack((x_vec, y_vec, z_vec)).T  # (201*201, 3)
    if i == 0:
        ref_pos = posvec
        den_vec = lden
    else:
        ref_pos = np.vstack((ref_pos, posvec))
        den_vec = np.hstack((den_vec, lden))

# add 12 more surfaces between each surface via interpolation
ldens2 = []
for i in range(12):
    theta = np.pi / 12 * (2 * i + 1)
    x_vec = x_plane.flatten() * np.cos(theta)
    y_vec = x_plane.flatten() * np.sin(theta)
    z_vec = y_plane.flatten()
    lden = (ldens[i] + ldens[(i + 1) % 12]) / 2
    lden = lden.flatten()
    ldens2.append(lden)
    posvec = np.vstack((x_vec, y_vec, z_vec)).T
    ref_pos = np.vstack((ref_pos, posvec))
    den_vec = np.hstack((den_vec, lden))

ldens12 = []
for i in range(12):
    ldens12.append(ldens[i])
    ldens12.append(ldens2[i])

# add 24 more surfaces between each surface via interpolation
ldens3 = []
for i in range(24):
    theta = np.pi / 24 * (2 * i + 1)
    x_vec = x_plane.flatten() * np.cos(theta)
    y_vec = x_plane.flatten() * np.sin(theta)
    z_vec = y_plane.flatten()
    lden = (ldens12[i] + ldens12[(i + 1) % 24]) / 2
    lden = lden.flatten()
    ldens3.append(lden)
    posvec = np.vstack((x_vec, y_vec, z_vec)).T
    ref_pos = np.vstack((ref_pos, posvec))
    den_vec = np.hstack((den_vec, lden))

ldens123 = []
for i in range(24):
    ldens123.append(ldens12[i])
    ldens123.append(ldens3[i])

# add 48 more surfaces between each surface via interpolation
ldens4 = []
for i in range(48):
    theta = np.pi / 48 * (2 * i + 1)
    x_vec = x_plane.flatten() * np.cos(theta)
    y_vec = x_plane.flatten() * np.sin(theta)
    z_vec = y_plane.flatten()
    lden = (ldens123[i] + ldens123[(i + 1) % 48]) / 2
    lden = lden.flatten()
    ldens4.append(lden)
    posvec = np.vstack((x_vec, y_vec, z_vec)).T
    ref_pos = np.vstack((ref_pos, posvec))
    den_vec = np.hstack((den_vec, lden))

In [ ]:
import plotly.graph_objects as go
import numpy as np

X = ref_pos[:, 0]
Y = ref_pos[:, 1]
Z = ref_pos[:, 2]
R = np.sqrt(X**2 + Y**2 + Z**2)
values = den_vec

no_axis = False

# extract points where R <= 10
indices = np.where(R <= 10)
X = X[indices]
Y = Y[indices]
Z = Z[indices]
values = values[indices]

# 3d scatter plot
inv = 8
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=X[::inv],
            y=Y[::inv],
            z=Z[::inv],
            mode="markers",
            marker=dict(
                size=1.2,
                color=values[::inv],  # set color to an array/list of desired values
                colorscale="Viridis",  # choose a colorscale
                opacity=0.1,
            ),
        )
    ]
)

if no_axis:
    fig.update_layout(
        scene=dict(
            xaxis=dict(showgrid=False, showticklabels=False, title=""),
            yaxis=dict(showgrid=False, showticklabels=False, title=""),
            zaxis=dict(showgrid=False, showticklabels=False, title=""),
            aspectmode="cube",
        )
    )
    # no background
    fig.update_layout(
        scene=dict(
            xaxis=dict(showbackground=False),
            yaxis=dict(showbackground=False),
            zaxis=dict(showbackground=False),
        )
    )

fig.write_html(OUTPUT_DIR_CPP + "/gcpm_v24_meridian_3d.html")
fig.show()